# DDR Chart Generator — Training Notebook

**Before running: Runtime → Change runtime type → T4 GPU**

### Assumed setup
Your Google Drive already has StepMania packs uploaded at:
```
MyDrive/142B Group/ddc_project/data/raw/
  Some Pack Name/
    Song Name/
      Song.sm
      Song.mp3
  Another Pack/
    ...
```
Folder names don't matter. The code searches recursively for any folder
containing both a .sm file and an audio file (.mp3 / .ogg / .wav).

### Run order
**Every session:** re-run cells 1, 2, 3 (Colab VMs are wiped on disconnect).
**First time only:** run cells 4 onwards. Results save to Drive automatically.
**Resuming after disconnect:** re-run cells 1-3, then jump straight to cell 5 with `--curriculum_start` set to the last completed stage.

In [ ]:
# ── 1. Mount Google Drive ───────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p '/content/drive/MyDrive/142B Group/ddc_project/data/cache'
!mkdir -p '/content/drive/MyDrive/142B Group/ddc_project/data/raw'
!mkdir -p '/content/drive/MyDrive/142B Group/ddc_project/checkpoints'
!mkdir -p '/content/drive/MyDrive/142B Group/ddc_project/logs'
print('Drive mounted.')

In [ ]:
# ── 2. Clone repo and symlink data ──────────────────────────────────────
%cd /content
!git clone --branch helena https://github.com/alexseungum/ieor142b_project.git ddc-chart-gen
%cd /content/ddc-chart-gen

# Symlink Drive data folder so code finds it at 'data/'
!ln -sf '/content/drive/MyDrive/142B Group/ddc_project/data' data
print('Repo cloned and data symlinked.')

In [ ]:
!git log --oneline -1 && echo "---" && git log --oneline -1 origin/helena

In [ ]:
# ── 3. Install dependencies ─────────────────────────────────────────────
!pip install librosa soundfile torchaudio --quiet
print('Done.')

In [ ]:
# ── 4. Verify data ───────────────────────────────────────────────────────
import sys, re
from pathlib import Path
from collections import defaultdict

sys.path.insert(0, '/content/ddc-chart-gen')

DATA_ROOT = '/content/drive/MyDrive/142B Group/ddc_project/data/raw'
root = Path(DATA_ROOT)
audio_exts = {'.ogg', '.mp3', '.wav'}

def quick_difficulties(chart_path: str):
    """Fast grep for dance-single difficulty tags — no full parse needed."""
    try:
        content = open(chart_path, 'r', encoding='utf-8', errors='ignore').read()
        is_ssc = chart_path.endswith('.ssc')
        diffs = []
        if is_ssc:
            types = re.findall(r'#STEPSTYPE\s*:([^;]+);', content, re.IGNORECASE)
            diffd = re.findall(r'#DIFFICULTY\s*:([^;]+);', content, re.IGNORECASE)
            for t, d in zip(types, diffd):
                if 'single' in t.lower():
                    diffs.append(d.strip().lower())
        else:
            for m in re.finditer(r'#NOTES\s*:(.*?)(?=\n[^,\n]|\Z)', content,
                                  re.DOTALL | re.IGNORECASE):
                block = m.group(1)
                lines = [l.strip().rstrip(':') for l in block.split('\n')
                         if l.strip() and not l.strip().startswith('//')]
                if len(lines) >= 3 and 'single' in lines[0].lower():
                    diffs.append(lines[2].lower())
        return diffs
    except Exception:
        return []

all_sm  = list(root.rglob('*.sm'))
all_ssc = list(root.rglob('*.ssc'))
song_dirs = sorted({f.parent for f in all_sm + all_ssc})

print(f"{'='*60}")
print(f"DATASET SUMMARY")
print(f"{'='*60}")
print(f"Total song directories found : {len(song_dirs)}")
print(f"  .sm  files                 : {len(all_sm)}")
print(f"  .ssc files                 : {len(all_ssc)}")

no_audio     = []
usable_pairs = []
pack_stats   = defaultdict(lambda: {'songs': 0, 'sm': 0, 'ssc': 0, 'no_audio': 0,
                                     'difficulties': defaultdict(int)})

for song_dir in song_dirs:
    pack = song_dir.parent.name
    sm_files    = list(song_dir.glob('*.sm'))
    ssc_files   = list(song_dir.glob('*.ssc'))
    audio_files = [f for f in song_dir.iterdir() if f.suffix.lower() in audio_exts]
    chart_files = sm_files if sm_files else ssc_files
    fmt = 'sm' if sm_files else 'ssc'

    pack_stats[pack]['songs'] += 1
    pack_stats[pack][fmt] += 1

    if not audio_files or not chart_files:
        pack_stats[pack]['no_audio'] += 1
        no_audio.append(song_dir)
        continue

    chart_path = str(chart_files[0])
    usable_pairs.append((str(audio_files[0]), chart_path, fmt, pack))
    for diff in quick_difficulties(chart_path):
        pack_stats[pack]['difficulties'][diff] += 1

print(f"\nSongs with audio             : {len(usable_pairs)}")
print(f"Songs missing audio (skipped): {len(no_audio)}")
if no_audio:
    for d in no_audio[:5]:
        print(f"  - {d.relative_to(root)}")
    if len(no_audio) > 5:
        print(f"  ... and {len(no_audio)-5} more")

diff_order = ['beginner', 'easy', 'medium', 'hard', 'challenge']
print(f"\n{'='*60}")
print(f"BREAKDOWN BY PACK")
print(f"{'='*60}")
for pack, stats in sorted(pack_stats.items()):
    fmt_str = []
    if stats['sm']:  fmt_str.append(f"{stats['sm']} .sm")
    if stats['ssc']: fmt_str.append(f"{stats['ssc']} .ssc")
    no_aud = f"  ⚠ {stats['no_audio']} missing audio" if stats['no_audio'] else ""
    print(f"\n  [{pack}]  {stats['songs']} songs ({', '.join(fmt_str)}){no_aud}")
    diff_counts = [f"{d[:3].upper()}:{stats['difficulties'].get(d,0)}" for d in diff_order]
    print(f"    difficulties: {' | '.join(diff_counts)}")

print(f"\n{'='*60}")
print(f"OVERALL DIFFICULTY BREAKDOWN (usable songs only)")
print(f"{'='*60}")
total_diffs = defaultdict(int)
for stats in pack_stats.values():
    for d, n in stats['difficulties'].items():
        total_diffs[d] += n
for d in diff_order:
    print(f"  {d.capitalize():<12}: {total_diffs[d]} charts")

print(f"\nTotal usable song-audio pairs: {len(usable_pairs)}")
print(f"  .sm  : {sum(1 for _,_,f,_ in usable_pairs if f=='sm')}")
print(f"  .ssc : {sum(1 for _,_,f,_ in usable_pairs if f=='ssc')}")

print(f"\n{'='*60}")
print(f"SANITY CHECK (first usable song)")
print(f"{'='*60}")
if usable_pairs:
    from utils.data_utils import parse_chart_file, build_sample
    audio_path, chart_path, fmt, pack = usable_pairs[0]
    print(f"Song  : {Path(chart_path).parent.name}  [{fmt.upper()}]")
    print(f"Pack  : {pack}")
    sm_data = parse_chart_file(chart_path)
    print(f"Title : {sm_data['title']}")
    print(f"Charts: {[(c['difficulty'], c['meter']) for c in sm_data['charts'] if 'single' in c['chart_type'].lower()]}")
    sample = build_sample(audio_path, chart_path)
    if sample:
        T = sample['beat_frames'].shape[0]
        print(f"Timesteps    : {T}  (beat positions)")
        print(f"mel shape    : {sample['mel'].shape}  (mel_bins, audio_frames)")
        print(f"y shape      : {sample['y'].shape}  (timesteps, 4 arrows)")
        print(f"Step density : {(sample['y'].sum(-1) > 0).mean():.1%} of timesteps have a step")
    else:
        print("build_sample returned None — no dance-single chart found")
else:
    print("No usable pairs found — check your data folder structure")

In [ ]:
# ── 4b. Build cache ───────────────────────────────────────────────────────
# Builds base_train.pkl and base_val.pkl before training starts.
# Safe to re-run — skips if cache already exists.
# Run this after cell 4 (verify data).

import os, sys, pickle
from pathlib import Path
sys.path.insert(0, '/content/ddc-chart-gen')

DATA_ROOT = '/content/drive/MyDrive/142B Group/ddc_project/data/raw'
CACHE_DIR = '/content/drive/MyDrive/142B Group/ddc_project/data/cache'
os.makedirs(CACHE_DIR, exist_ok=True)

base_train_cache = f'{CACHE_DIR}/base_train.pkl'
base_val_cache   = f'{CACHE_DIR}/base_val.pkl'

# ── Check if cache already exists ────────────────────────────
if os.path.exists(base_train_cache) and os.path.exists(base_val_cache):
    print("Cache already exists — skipping build.")
    train_gb = os.path.getsize(base_train_cache) / 1e9
    val_gb   = os.path.getsize(base_val_cache)   / 1e9
    with open(base_train_cache, 'rb') as f:
        n = len(pickle.load(f))
    print(f"  base_train.pkl : {train_gb:.2f} GB")
    print(f"  base_val.pkl   : {val_gb:.2f} GB")
    print(f"  {n} song-difficulty samples cached")
else:
    from dataset import _process_song
    root = Path(DATA_ROOT)
    audio_exts = {'.ogg', '.mp3', '.wav'}

    # Find all usable song pairs
    all_sm  = list(root.rglob('*.sm'))
    all_ssc = list(root.rglob('*.ssc'))
    song_dirs = sorted({f.parent for f in all_sm + all_ssc})

    pairs = []
    for song_dir in song_dirs:
        sm_files    = list(song_dir.glob('*.sm'))
        ssc_files   = list(song_dir.glob('*.ssc'))
        audio_files = [f for f in song_dir.iterdir() if f.suffix.lower() in audio_exts]
        chart_files = sm_files if sm_files else ssc_files
        if chart_files and audio_files:
            pairs.append((str(audio_files[0]), str(chart_files[0])))

    print(f"Building cache for {len(pairs)} songs...")
    print("This takes ~10-20 min on first run. Progress updates every 10 songs.")

    # Process with imap_unordered so one bad song doesn't kill everything.
    # Use 2 workers — Drive I/O is the bottleneck, not CPU.
    from multiprocessing import Pool
    samples = []
    failed  = 0
    N_WORKERS = 2

    with Pool(N_WORKERS) as pool:
        for i, result in enumerate(pool.imap_unordered(_process_song, pairs)):
            samples.extend(result)
            if not result:
                failed += 1
            if (i + 1) % 10 == 0 or (i + 1) == len(pairs):
                print(f"  [{i+1}/{len(pairs)}] {len(samples)} samples built"
                      + (f"  ({failed} skipped)" if failed else ""))

    print(f"\nDone. {len(samples)} song-difficulty samples total.")
    if failed:
        print(f"  {failed} songs skipped (corrupt audio/chart or no dance-single chart)")

    # Save as both train and val base caches — same data, split happens at load time
    print("Saving to Drive...")
    with open(base_train_cache, 'wb') as f:
        pickle.dump(samples, f)
    with open(base_val_cache, 'wb') as f:
        pickle.dump(samples, f)
    print(f"Saved: {os.path.getsize(base_train_cache)/1e9:.2f} GB per file")
    print("Ready to train.")

In [ ]:
# ── 5. Train ─────────────────────────────────────────────────────────────
import os
DATA_ROOT = '/content/drive/MyDrive/142B Group/ddc_project/data/raw'
CACHE_DIR = '/content/drive/MyDrive/142B Group/ddc_project/data/cache'
CKPT_DIR  = '/content/drive/MyDrive/142B Group/ddc_project/checkpoints'
os.environ['DATA_ROOT'] = DATA_ROOT
os.environ['CACHE_DIR'] = CACHE_DIR
os.environ['CKPT_DIR']  = CKPT_DIR

%cd /content/ddc-chart-gen
!python train.py \
    --data_root      "$DATA_ROOT" \
    --cache_dir      "$CACHE_DIR" \
    --checkpoint_dir "$CKPT_DIR" \
    --num_workers 0

In [ ]:
# ── 5b. Resume after disconnect ──────────────────────────────────────────
import torch
CKPT_DIR = '/content/drive/MyDrive/142B Group/ddc_project/checkpoints'
ckpt = torch.load(f'{CKPT_DIR}/best_model.pt', map_location='cpu')
print(f"Last saved: stage {ckpt['stage']}, epoch {ckpt['epoch']}, val F1 {ckpt['val_f1']:.4f}")
print(f"Rerun cell 5 — curriculum_start is read from config.py (currently set to {ckpt['stage']})")

In [ ]:
# ── 6. Plot loss curves ──────────────────────────────────────────────────
%cd /content/ddc-chart-gen
import json
import matplotlib.pyplot as plt

with open('logs/train_history.json') as f:
    history = json.load(f)

train_h = history['train']
val_h   = history['val']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
stage_colors = {0:'#3498db', 1:'#2ecc71', 2:'#f39c12', 3:'#e74c3c', 4:'#9b59b6'}
stage_names  = {0:'Beginner', 1:'Easy', 2:'Medium', 3:'Hard', 4:'Challenge'}

for metric, ax, title in [
    ('loss',       axes[0], 'Total Loss'),
    ('step_loss',  axes[1], 'Step Placement Loss'),
    ('f1',         axes[2], 'Step F1 Score'),
]:
    for stage in range(5):
        t_vals = [x[metric] for x in train_h if x['stage'] == stage]
        v_vals = [x[metric] for x in val_h   if x['stage'] == stage]
        if not t_vals:
            continue
        color = stage_colors[stage]
        ax.plot(t_vals, color=color, alpha=0.9, label=f'{stage_names[stage]} train')
        ax.plot(v_vals, color=color, alpha=0.5, linestyle='--', label=f'{stage_names[stage]} val')
    ax.set_title(title)
    ax.set_xlabel('Epoch within stage')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=7)

plt.suptitle('Training curves by curriculum stage', y=1.02)
plt.tight_layout()
plt.savefig('logs/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
!mkdir -p '/content/drive/MyDrive/142B Group/ddc_project/logs'
!cp logs/training_curves.png '/content/drive/MyDrive/142B Group/ddc_project/logs/'
print('Saved to Drive.')

In [ ]:
# ── 6b. Seeded validation generation ───────────────────────────────────────
# Loads a real val song from cache, seeds decoder with first 16 GT steps,
# then generates autoregressively. Diagnostic: good output = cold-start problem;
# bad output = training metrics are misleading.
%cd /content/ddc-chart-gen
import torch, numpy as np, matplotlib.pyplot as plt
import pickle, random, base64, sys, os
from pathlib import Path
sys.path.insert(0, '/content/ddc-chart-gen')
from models.model import DDRTransformer
from config import SEQ_LEN, CONTEXT_FRAMES, N_MELS
from visualizer import build_html
from google.colab import files

CKPT_DIR   = '/content/drive/MyDrive/142B Group/ddc_project/checkpoints'
DATA_ROOT  = '/content/drive/MyDrive/142B Group/ddc_project/data/raw'
CACHE_DIR  = '/content/drive/MyDrive/142B Group/ddc_project/data/cache'
TARGET_DIFF = 2    # 0=Beginner 1=Easy 2=Medium 3=Hard 4=Challenge
THRESHOLD   = 0.5
N_SEED      = 16   # number of GT steps used as seed history
DIFF_NAMES  = {0:'Beginner',1:'Easy',2:'Medium',3:'Hard',4:'Challenge'}

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# ── Load model ───────────────────────────────────────────────────────────────
ckpt = torch.load(f'{CKPT_DIR}/best_model.pt', map_location=device)
model_args = ckpt.get('args', {})
model = DDRTransformer(
    d_model=model_args.get('d_model', 256),
    nhead=model_args.get('nhead', 8),
    num_encoder_layers=model_args.get('n_layers', 4),
    dim_feedforward=model_args.get('d_ff', 1024),
    dropout=0.0,
).to(device)
model.load_state_dict(ckpt['model_state'], strict=False)
model.eval()
print(f"Loaded: stage {ckpt.get('stage','?')}, epoch {ckpt.get('epoch','?')}")

# ── Load val songs directly from base cache ──────────────────────────────────
with open(f'{CACHE_DIR}/base_val.pkl', 'rb') as f:
    all_val = pickle.load(f)

candidates = [s for s in all_val if s['difficulty'] == TARGET_DIFF]
if not candidates:
    candidates = all_val
    print(f'No songs at difficulty {TARGET_DIFF}, using all val songs')
song = random.choice(candidates)
title = song.get('title', 'unknown')
print(f"Song: '{title}'  difficulty={song['difficulty']}  total_timesteps={song['y'].shape[0]}")

# ── Process full song in SEQ_LEN chunks ─────────────────────────────────────
mel   = song['mel']
bf    = song['beat_frames']
y_np  = song['y'].astype(np.float32)
st    = song['subdiv_types']
T_total = (len(bf) // SEQ_LEN) * SEQ_LEN  # trim to whole chunks
n_chunks = T_total // SEQ_LEN
print(f"Song length: {len(bf)} timesteps → {n_chunks} chunks of {SEQ_LEN}")

ctx     = CONTEXT_FRAMES
mel_f32 = np.pad(mel.astype(np.float32), ((0, 0), (ctx, ctx)))
diff_dev = torch.tensor([song['difficulty']], dtype=torch.long, device=device)

step_mask_all  = np.zeros(T_total, dtype=bool)
step_probs_all = np.zeros(T_total)
arrows_all     = np.zeros((T_total, 4), dtype=np.float32)
seed_cutoff_global = 0  # end of seed region (chunk 0 only)

for chunk_idx in range(n_chunks):
    s = chunk_idx * SEQ_LEN
    e = s + SEQ_LEN
    bf_c  = bf[s:e]
    y_c   = y_np[s:e]
    st_c  = st[s:e]

    col_idx = bf_c[:, None] + np.arange(-ctx, ctx + 1)[None, :]
    X_np    = mel_f32[:, col_idx].transpose(1, 2, 0)
    X_dev   = torch.from_numpy(X_np).unsqueeze(0).to(device)
    st_dev  = torch.from_numpy(st_c).unsqueeze(0).to(device)

    arrows = torch.zeros(1, SEQ_LEN, 4, device=device)

    if chunk_idx == 0:
        # Seed first chunk with first N_SEED GT steps
        gt_step_pos = np.where(y_c.sum(-1) > 0)[0]
        seed_pos    = gt_step_pos[:N_SEED]
        seed_cutoff = int(seed_pos[-1]) + 1 if len(seed_pos) >= N_SEED else 0
        seed_cutoff_global = seed_cutoff
        for p in seed_pos:
            arrows[0, p, :] = torch.from_numpy(y_c[p]).to(device)
            step_mask_all[s + p] = True
            arrows_all[s + p] = y_c[p]
        gen_start = seed_cutoff
        print(f"Chunk 0: seeding positions {seed_pos.tolist()}, generating from t={seed_cutoff}")
    else:
        gen_start = 0

    with torch.no_grad():
        encoder_out = model.encode(X_dev, diff_dev, st_dev)
        for t in range(gen_start, SEQ_LEN):
            sl, al = model.decoder(arrows, encoder_out)
            prob = torch.sigmoid(sl[0, t, 0]).item()
            step_probs_all[s + t] = prob
            if prob > THRESHOLD:
                step_mask_all[s + t] = True
                ap   = torch.sigmoid(al[0, t, :])
                pred = (ap > THRESHOLD).float()
                if pred.sum() == 0:
                    pred[ap.argmax()] = 1.0
                arrows[0, t, :] = pred
                arrows_all[s + t] = pred.cpu().numpy()

T          = T_total
y_np       = y_np[:T_total]
step_mask  = step_mask_all
step_probs = step_probs_all
arrow_preds = arrows_all.astype(int)
arrow_preds[~step_mask] = 0
seed_cutoff = seed_cutoff_global
print(f"Total: {step_mask.sum()} steps predicted over {T} timesteps ({100*step_mask.mean():.1f}% density)")

# ── Metrics on generated portion ────────────────────────────────────────────
gt_gen   = y_np[seed_cutoff:].sum(-1) > 0
pred_gen = step_mask[seed_cutoff:]
tp = (gt_gen & pred_gen).sum()
fp = (~gt_gen & pred_gen).sum()
fn = (gt_gen & ~pred_gen).sum()
prec = tp / (tp + fp + 1e-8)
rec  = tp / (tp + fn + 1e-8)
f1   = 2 * prec * rec / (prec + rec + 1e-8)
print(f"Generated [{seed_cutoff}:{T}]: {pred_gen.sum()} predicted  {gt_gen.sum()} GT")
print(f"  F1={f1:.3f}  Precision={prec:.3f}  Recall={rec:.3f}")

# ── Plots ────────────────────────────────────────────────────────────────────
ARROW_COLORS = ['#e74c3c','#3498db','#2ecc71','#f39c12']
ARROW_NAMES  = ['L','D','U','R']

fig, axes = plt.subplots(3, 1, figsize=(16, 8))

ax = axes[0]
ax.axvspan(0, seed_cutoff, alpha=0.12, color='green', label=f'Seed ({N_SEED} GT steps)')
ax.plot(step_probs, lw=0.6, color='#2980b9', label='Step prob')
ax.axhline(THRESHOLD, color='red', ls='--', lw=0.8, label='Threshold')
ax.set_ylabel('Step probability'); ax.set_title(f"Step probs — '{title}'")
ax.legend(fontsize=8); ax.grid(True, alpha=0.2)

ax = axes[1]
ax.set_title('Ground truth')
for i,(c,n) in enumerate(zip(ARROW_COLORS, ARROW_NAMES)):
    ts = np.where(y_np[:,i]>0)[0]
    ax.scatter(ts, np.full_like(ts,i), c=c, s=4, label=n)
ax.set_yticks([0,1,2,3]); ax.set_yticklabels(ARROW_NAMES)
ax.set_xlim(0,T); ax.grid(True,alpha=0.2); ax.legend(fontsize=8)

ax = axes[2]
ax.axvspan(0, seed_cutoff, alpha=0.12, color='green')
ax.set_title('Predicted (green=seed, white=generated)')
for i,(c,n) in enumerate(zip(ARROW_COLORS, ARROW_NAMES)):
    ts = np.where(arrow_preds[:,i]>0)[0]
    ax.scatter(ts, np.full_like(ts,i), c=c, s=4, label=n)
ax.set_yticks([0,1,2,3]); ax.set_yticklabels(ARROW_NAMES)
ax.set_xlim(0,T); ax.grid(True,alpha=0.2); ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('logs/seeded_generation.png', dpi=150)
plt.show()

# ── Find audio file and read BPM + offset from SM ────────────────────────────
audio_data_uri = None
audio_exts = {'.mp3', '.ogg', '.wav'}
mime_map   = {'.mp3':'audio/mpeg', '.ogg':'audio/ogg', '.wav':'audio/wav'}

def _norm(s): return s.lower().strip()

song_bpm    = 120.0
song_offset = 0.0
print(f"Searching for audio matching title: '{title}'")
for sm_path in Path(DATA_ROOT).rglob('*.[sS][mM]'):
    try:
        text = sm_path.read_text(errors='ignore')
        for line in text.splitlines():
            if line.upper().startswith('#TITLE:'):
                sm_title = line.split(':', 1)[1].rstrip(';').strip()
                if _norm(sm_title) == _norm(title):
                    for tag_line in text.splitlines():
                        tag_upper = tag_line.upper().lstrip()
                        if tag_upper.startswith('#BPMS:'):
                            try:
                                song_bpm = float(tag_line.split(':',1)[1].split('=')[1].rstrip(';').split(',')[0].strip())
                            except Exception:
                                pass
                        elif tag_upper.startswith('#OFFSET:'):
                            try:
                                song_offset = float(tag_line.split(':',1)[1].rstrip(';').strip())
                            except Exception:
                                pass
                    for f in sm_path.parent.iterdir():
                        if f.suffix.lower() in audio_exts:
                            mime = mime_map[f.suffix.lower()]
                            audio_data_uri = f"data:{mime};base64," + base64.b64encode(f.read_bytes()).decode()
                            print(f"Found audio: {f.name}  BPM={song_bpm:.2f}  offset={song_offset:.4f}s")
                            break
                break
    except Exception:
        pass
    if audio_data_uri:
        break

if not audio_data_uri:
    words = [w for w in _norm(title).split() if len(w) > 3]
    for song_dir in Path(DATA_ROOT).rglob('*'):
        if not song_dir.is_dir():
            continue
        if any(w in _norm(song_dir.name) for w in words):
            for f in song_dir.iterdir():
                if f.suffix.lower() in audio_exts:
                    mime = mime_map[f.suffix.lower()]
                    audio_data_uri = f"data:{mime};base64," + base64.b64encode(f.read_bytes()).decode()
                    print(f"Found via folder '{song_dir.name}': {f.name}  BPM={song_bpm:.2f}  offset={song_offset:.4f}s")
                    break
        if audio_data_uri:
            break

if not audio_data_uri:
    print(f"Audio not found. Sample SM titles found:")
    for i, sm_path in enumerate(Path(DATA_ROOT).rglob('*.[sS][mM]')):
        try:
            for line in sm_path.read_text(errors='ignore').splitlines():
                if line.upper().startswith('#TITLE:'):
                    print(f"  '{line.split(':',1)[1].rstrip(';').strip()}'")
                    break
        except Exception:
            pass
        if i >= 9:
            break
    print("Visualizer will load without playback")

# ── Build and download visualizer ────────────────────────────────────────────
events = [
    {'t': int(t), 'arrows': [int(arrow_preds[t,i]) for i in range(4)]}
    for t in range(T) if step_mask[t]
]
chart_data = {
    'title':           title,
    'bpm':             song_bpm,
    'offset':          song_offset,
    'difficulty':      DIFF_NAMES.get(int(song['difficulty']), 'Medium'),
    'meter':           int(song['difficulty']) * 3 + 3,
    'subdivision':     24,
    'total_steps':     int(step_mask.sum()),
    'total_timesteps': T,
    'events':          events,
}
html = build_html(chart_data, audio_data_uri=audio_data_uri)
with open('logs/seeded_visualizer.html', 'w') as f:
    f.write(html)
files.download('logs/seeded_visualizer.html')
print(f"Downloaded visualizer — {len(events)} steps  offset={song_offset:.4f}s")


In [ ]:
# ── 6c. Visualize a GT chart by song title ──────────────────────────────────
# Set SONG_TITLE (case-insensitive). Optionally set PACK_NAME to search faster.
# Leave PACK_NAME = '' to search all packs.
%cd /content/ddc-chart-gen
import base64, sys
from pathlib import Path
sys.path.insert(0, '/content/ddc-chart-gen')
from utils.data_utils import parse_chart_file
from visualizer import build_chart_json, build_html
from google.colab import files

DATA_ROOT     = '/content/drive/MyDrive/142B Group/ddc_project/data/raw'
PACK_NAME     = ''                      # <-- e.g. 'DDR A' or '' for all packs
SONG_TITLE    = 'PUT SONG TITLE HERE'   # <-- edit this (case-insensitive)
GT_DIFFICULTY = None                    # e.g. 'medium', or None for first chart found

audio_exts = {'.mp3', '.ogg', '.wav'}
mime_map   = {'.mp3':'audio/mpeg', '.ogg':'audio/ogg', '.wav':'audio/wav'}
def _norm(s): return s.lower().strip()

search_root = Path(DATA_ROOT)
if PACK_NAME:
    candidates = [p for p in search_root.iterdir()
                  if p.is_dir() and _norm(p.name) == _norm(PACK_NAME)]
    if candidates:
        search_root = candidates[0]
        print(f"Searching in pack: {search_root.name}")
    else:
        print(f"Pack '{PACK_NAME}' not found, searching all packs")

all_charts = (list(search_root.rglob('*.sm'))  + list(search_root.rglob('*.SM')) +
              list(search_root.rglob('*.ssc')) + list(search_root.rglob('*.SSC')))
print(f"Scanning {len(all_charts)} chart files...")

match = None
for chart_path in all_charts:
    try:
        sm_data = parse_chart_file(str(chart_path))
        if _norm(sm_data['title']) == _norm(SONG_TITLE):
            match = (chart_path, sm_data)
            break
    except Exception:
        pass

if match is None:
    print(f"'{SONG_TITLE}' not found. Available titles:")
    seen = set()
    for chart_path in all_charts:
        try:
            title = parse_chart_file(str(chart_path))['title']
            if title not in seen:
                print(f"  '{title}'")
                seen.add(title)
        except Exception:
            pass
else:
    chart_path, sm_data = match
    print(f"Found: {chart_path}")
    diffs = [c['difficulty'] for c in sm_data['charts'] if 'single' in c['chart_type'].lower()]
    print(f"  Available difficulties: {diffs}")

    chart_data = build_chart_json(str(chart_path), difficulty_filter=GT_DIFFICULTY)
    print(f"  Diff: {chart_data['difficulty']}  BPM: {chart_data['bpm']:.1f}  "
          f"Offset: {chart_data['offset']:.4f}s  Steps: {chart_data['total_steps']}")

    audio_data_uri = None
    for f in chart_path.parent.iterdir():
        if f.suffix.lower() in audio_exts:
            mime = mime_map[f.suffix.lower()]
            audio_data_uri = f"data:{mime};base64," + base64.b64encode(f.read_bytes()).decode()
            print(f"  Audio: {f.name}")
            break
    if not audio_data_uri:
        print("  Audio not found — visualizer loads without playback")

    html = build_html(chart_data, audio_data_uri=audio_data_uri)
    out  = 'logs/gt_visualizer.html'
    with open(out, 'w') as f:
        f.write(html)
    files.download(out)
    print(f"Downloaded {out}")


In [ ]:
# ── 7. Generate a chart ──────────────────────────────────────────────────
%cd /content/ddc-chart-gen
import os
from pathlib import Path

# Option A: pick a song from the dataset (uncomment)
# DATA_ROOT = '/content/drive/MyDrive/142B Group/ddc_project/data/raw'
# all_charts = list(Path(DATA_ROOT).rglob('*.sm')) + list(Path(DATA_ROOT).rglob('*.ssc'))
# for chart in all_charts:
#     audio = [f for f in chart.parent.iterdir()
#              if f.suffix.lower() in {'.ogg', '.mp3', '.wav'}]
#     if audio:
#         test_audio = str(audio[0])
#         print(f'Using: {chart.parent.name} — {audio[0].name}')
#         break

# Option B: upload any song from your computer (mp3, ogg, wav)
from google.colab import files as colab_files
uploaded = colab_files.upload()
test_audio = list(uploaded.keys())[0]
print(f'Using: {test_audio}')

# ── Set BPM manually — check the song's .sm/.ssc file or use a BPM finder ──
SONG_BPM = 123.0   # <-- edit this

CKPT_DIR = '/content/drive/MyDrive/142B Group/ddc_project/checkpoints'
os.environ['TEST_AUDIO'] = test_audio
os.environ['CKPT_DIR']   = CKPT_DIR
os.environ['SONG_BPM']   = str(SONG_BPM)

# Adjust --difficulty (0-4) and --threshold (0.1-0.9, lower = more arrows)
!python generate.py \
    --audio      "$TEST_AUDIO" \
    --checkpoint "$CKPT_DIR/best_model.pt" \
    --bpm        "$SONG_BPM" \
    --difficulty 2 \
    --threshold  0.5 \
    --output     output_chart

!cp -r output_chart '/content/drive/MyDrive/142B Group/ddc_project/'
print('Done! Run cell 9 to download visualizer.html')

In [ ]:
# ── 8. Threshold sweep ───────────────────────────────────────────────────
%cd /content/ddc-chart-gen
import torch, os
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '/content/ddc-chart-gen')

from models.model import DDRTransformer, generate_chart
from generate import audio_to_model_input

CKPT_DIR = '/content/drive/MyDrive/142B Group/ddc_project/checkpoints'
ckpt = torch.load(f'{CKPT_DIR}/best_model.pt', map_location='cpu')
model_args = ckpt.get('args', {})
model = DDRTransformer(
    d_model=model_args.get('d_model', 256),
    nhead=model_args.get('nhead', 8),
    num_encoder_layers=model_args.get('n_layers', 4),
    dim_feedforward=model_args.get('d_ff', 1024),
    dropout=0.0,
)
model.load_state_dict(ckpt['model_state'])
X, subdiv_types = audio_to_model_input(test_audio, bpm=float(os.environ['SONG_BPM']))

thresholds = np.linspace(0.1, 0.9, 17)
densities  = []
for th in thresholds:
    mask, _ = generate_chart(model, X, subdiv_types, difficulty=2, step_threshold=float(th),device='cuda')
    densities.append(mask.mean())

plt.figure(figsize=(7, 4))
plt.plot(thresholds, densities, 'o-', color='#e74c3c')
plt.axvline(0.5, color='gray', linestyle='--', alpha=0.5, label='default threshold')
plt.xlabel('Step threshold')
plt.ylabel('Fraction of timesteps with a step')
plt.title('Difficulty knob: threshold vs. chart density')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('logs/threshold_sweep.png', dpi=150)
plt.show()
!mkdir -p '/content/drive/MyDrive/142B Group/ddc_project/logs'
!cp logs/threshold_sweep.png '/content/drive/MyDrive/142B Group/ddc_project/logs/'

In [ ]:
# ── 8b. Sanity check: step & arrow probability distributions ────────────
%cd /content/ddc-chart-gen
import torch, numpy as np, matplotlib.pyplot as plt
import sys, os
sys.path.insert(0, "/content/ddc-chart-gen")

from models.model import DDRTransformer
from generate import audio_to_model_input

device = "cuda" if torch.cuda.is_available() else "cpu"

CKPT_DIR = "/content/drive/MyDrive/142B Group/ddc_project/checkpoints"
ckpt = torch.load(f"{CKPT_DIR}/best_model.pt", map_location=device)
model_args = ckpt.get("args", {})
model = DDRTransformer(
    d_model=model_args.get("d_model", 256),
    nhead=model_args.get("nhead", 8),
    num_encoder_layers=model_args.get("n_layers", 4),
    dim_feedforward=model_args.get("d_ff", 1024),
    dropout=0.0,
).to(device)
model.load_state_dict(ckpt["model_state"], strict=False)
model.eval()

X, subdiv_types = audio_to_model_input(test_audio, bpm=float(os.environ["SONG_BPM"]))
X = X.to(device)
subdiv_types = subdiv_types.to(device)
diff_tensor = torch.tensor([2], dtype=torch.long, device=device)  # difficulty=2

with torch.no_grad():
    encoder_out = model.encode(X, diff_tensor, subdiv_types)       # (1, T, d_model)
    B, T, _ = encoder_out.shape
    arrows_zeros = torch.zeros(B, T, 4, device=device)
    step_logits, arrow_logits = model.decoder(arrows_zeros, encoder_out)

step_probs  = torch.sigmoid(step_logits.squeeze(-1).squeeze(0)).cpu().numpy()  # (T,)
arrow_probs = torch.sigmoid(arrow_logits.squeeze(0)).cpu().numpy()              # (T, 4)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# 1. Step probability over time
ax = axes[0]
ax.plot(np.arange(len(step_probs)), step_probs, linewidth=0.5, color="#2980b9", alpha=0.8)
ax.axhline(0.5, color="red", linestyle="--", linewidth=0.8, label="threshold=0.5")
ax.set_xlabel("Timestep")
ax.set_ylabel("Step probability")
ax.set_title("Step placement probability across song")
ax.legend()
ax.grid(True, alpha=0.2)

# 2. Arrow probability distribution per direction
ARROW_NAMES  = ["Left", "Down", "Up", "Right"]
ARROW_COLORS = ["#e74c3c", "#3498db", "#2ecc71", "#f39c12"]
ax = axes[1]
bins = np.linspace(0, 1, 25)
for i, (name, color) in enumerate(zip(ARROW_NAMES, ARROW_COLORS)):
    ax.hist(arrow_probs[:, i], bins=bins, alpha=0.5, color=color, label=name)
ax.set_xlabel("Arrow probability")
ax.set_ylabel("Count")
ax.set_title("Arrow probability distribution by direction")
ax.legend()
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig("logs/prob_sanity.png", dpi=150)
plt.show()
print(f"Step prob — mean: {step_probs.mean():.3f}, median: {np.median(step_probs):.3f}, >0.5: {(step_probs>0.5).mean()*100:.1f}%")
for i, name in enumerate(ARROW_NAMES):
    print(f"  {name}: mean prob {arrow_probs[:, i].mean():.3f}")


In [ ]:
# ── 9. Download results ──────────────────────────────────────────────────
%cd /content/ddc-chart-gen
CKPT_DIR = '/content/drive/MyDrive/142B Group/ddc_project/checkpoints'

from google.colab import files
files.download('output_chart/visualizer.html')
files.download('output_chart/chart.sm')
files.download(f'{CKPT_DIR}/best_model.pt')
files.download('logs/training_curves.png')
files.download('logs/threshold_sweep.png')